# Sentiment Analysis of Customer Reviews
## DLBDSEAIS02 — Project: Artificial Intelligence — Task 2

This notebook documents the full development process of a sentiment analysis system for customer product reviews.

The system classifies reviews into three sentiment classes — **negative**, **neutral**, and **positive** — using a pretrained HuggingFace BERT model. Results are stored in a SQLite database, exposed via a FastAPI REST API, and visualised in a Streamlit dashboard.

### Notebook Structure
1. Environment verification
2. Model loading and single-review classification
3. Database initialisation
4. API integration tests
5. Dataset loading and preprocessing (Amazon Customer Reviews)
6. Batch evaluation on a stratified sample
7. Quality monitoring and retraining pipeline
8. Summary and limitations

---
## 1. Environment Verification

Verify that all required packages are installed before proceeding.

In [ ]:
import sys
import importlib

print(f"Python version: {sys.version}\n")

required_packages = ['transformers', 'fastapi', 'uvicorn', 'streamlit', 'sqlalchemy', 'pandas', 'requests', 'tqdm']

for package in required_packages:
    try:
        importlib.import_module(package)
        print(f"  {package}: OK")
    except ImportError:
        print(f"  {package}: MISSING — run: pip install {package}")

---
## 2. Model Loading and Single-Review Classification

### Model choice
The selected model is `nlptown/bert-base-multilingual-uncased-sentiment` — a BERT model fine-tuned on product reviews in six languages. It outputs a star rating (1–5), which is then mapped to three sentiment classes:

| Stars | Sentiment |
|-------|-----------|
| 1–2   | negative  |
| 3     | neutral   |
| 4–5   | positive  |

A pretrained model was chosen over training from scratch because it provides strong out-of-the-box performance on product review data without requiring labelled training data or significant compute resources.

In [ ]:
import os
os.chdir(r'C:\Users\Gebruiker\Project AI')

from transformers import pipeline

# Load model — downloads weights on first run (~700 MB), cached afterwards
classifier = pipeline(
    "text-classification",
    model="nlptown/bert-base-multilingual-uncased-sentiment"
)

print("Model loaded successfully.")

In [ ]:
def stars_to_sentiment(label: str) -> str:
    """Map star rating label to a three-class sentiment string."""
    stars = int(label[0])
    if stars <= 2:
        return 'negative'
    elif stars == 3:
        return 'neutral'
    else:
        return 'positive'


def predict_sentiment(text: str) -> dict:
    """Run the full classification pipeline on a single review text."""
    result = classifier(text, truncation=True, max_length=512)[0]
    return {
        'text':       text,
        'sentiment':  stars_to_sentiment(result['label']),
        'stars':      result['label'],
        'confidence': round(result['score'], 4)
    }


# --- Single-review tests ---
test_reviews = [
    "This product is absolutely amazing, best purchase ever!",
    "It was okay, nothing special — does the job.",
    "Complete waste of money. Broke after two days."
]

for review in test_reviews:
    result = predict_sentiment(review)
    print(f"[{result['sentiment'].upper():8}] ({result['confidence']:.0%}) {result['text'][:60]}")

All three sentiment classes are correctly identified with high confidence. The model handles both clear positive and negative language as well as ambiguous neutral phrasing.

---
## 3. Database Initialisation

A SQLite database stores all predictions alongside human-verified labels. SQLite was chosen for its zero-configuration setup and portability — appropriate for a single-server deployment.

In [ ]:
import sqlite3
import pandas as pd
from datetime import datetime

DB_PATH = 'reviews.db'

def init_db(db_path: str = DB_PATH) -> None:
    """Create the reviews table if it does not already exist."""
    conn = sqlite3.connect(db_path)
    conn.execute("""
        CREATE TABLE IF NOT EXISTS reviews (
            id                 INTEGER PRIMARY KEY AUTOINCREMENT,
            text               TEXT    NOT NULL,
            sentiment          TEXT    NOT NULL,
            stars              TEXT    NOT NULL,
            confidence         REAL    NOT NULL,
            verified_sentiment TEXT,
            timestamp          TEXT    NOT NULL
        )
    """)
    conn.commit()
    conn.close()


def save_prediction(prediction: dict, db_path: str = DB_PATH) -> None:
    """Persist a single prediction to the database."""
    conn = sqlite3.connect(db_path)
    conn.execute(
        "INSERT INTO reviews (text, sentiment, stars, confidence, timestamp) VALUES (?, ?, ?, ?, ?)",
        (prediction['text'], prediction['sentiment'], prediction['stars'],
         prediction['confidence'], datetime.now().isoformat())
    )
    conn.commit()
    conn.close()


# Initialise database and insert the three test predictions
init_db()
for review in test_reviews:
    save_prediction(predict_sentiment(review))

# Verify contents
conn = sqlite3.connect(DB_PATH)
df_db = pd.read_sql('SELECT * FROM reviews ORDER BY id DESC LIMIT 5', conn)
conn.close()
df_db

The `verified_sentiment` column is initially NULL. It is populated by the analyst via the `/correct` API endpoint, forming the basis of the human-in-the-loop workflow.

---
## 4. API Integration Tests

The REST API (`api.py`) must be running before executing this section.

**Start the API in a separate terminal:**
```bash
cd "C:\Users\Gebruiker\Project AI"
uvicorn api:app --port 8000
```

In [ ]:
import requests

API_URL = "http://localhost:8000"

# Health check
response = requests.get(f"{API_URL}/health")
print("Health:", response.json())

# Predict endpoint
response = requests.post(
    f"{API_URL}/predict",
    json={"text": "Absolutely fantastic product, would buy again!"}
)
print("Predict:", response.json())

# Correct endpoint (human-in-the-loop)
response = requests.post(
    f"{API_URL}/correct",
    json={"review_id": 1, "verified_sentiment": "positive"}
)
print("Correct:", response.json())

# Quality check
response = requests.get(f"{API_URL}/quality")
print("Quality:", response.json())

All endpoints respond correctly. The full API documentation (OpenAPI / Swagger UI) is available at `http://localhost:8000/docs` when the server is running.

---
## 5. Dataset Loading and Preprocessing

### Dataset
The **Amazon Customer Reviews** dataset (Kaggle, 34,660 reviews) is used for evaluation. Ground-truth sentiment labels are derived from the star rating field — no manual annotation is required.

### Class imbalance
The dataset is heavily skewed toward positive reviews, reflecting typical Amazon rating distributions. Stratified sampling is applied during evaluation to ensure equal representation of all three classes.

In [ ]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Load dataset
DATASET_PATH = r'C:\Users\Gebruiker\AmazonProductReviews.csv'
df_raw = pd.read_csv(DATASET_PATH, low_memory=False)

print(f"Raw dataset: {df_raw.shape[0]:,} rows, {df_raw.shape[1]} columns")
print(f"Relevant columns: {['reviews.text', 'reviews.rating']}")

In [ ]:
# Extract and clean relevant columns
df_amazon = df_raw[['reviews.text', 'reviews.rating']].dropna().copy()
df_amazon.columns = ['text', 'rating']
df_amazon['rating'] = df_amazon['rating'].astype(int)

# Derive ground-truth sentiment from star rating
def rating_to_sentiment(rating: int) -> str:
    """Convert a 1-5 star rating to a three-class sentiment label."""
    if rating <= 2:
        return 'negative'
    elif rating == 3:
        return 'neutral'
    else:
        return 'positive'

df_amazon['true_sentiment'] = df_amazon['rating'].apply(rating_to_sentiment)

print("Class distribution (full dataset):")
print(df_amazon['true_sentiment'].value_counts().to_string())
print(f"\nNote: dataset is heavily imbalanced — positive reviews dominate ({df_amazon['true_sentiment'].eq('positive').mean():.1%})")

In [ ]:
# Stratified sample: 100 reviews per class
# Equal class sizes eliminate imbalance bias in evaluation metrics
df_sample = (
    df_amazon
    .groupby('true_sentiment', group_keys=False)
    .apply(lambda x: x.sample(min(len(x), 100), random_state=42))
    .reset_index(drop=True)
)

print(f"Stratified sample: {len(df_sample)} reviews")
print(df_sample['true_sentiment'].value_counts().to_string())

---
## 6. Batch Evaluation

The pretrained model is evaluated on the stratified sample. Each review is classified independently; ground-truth labels (derived from star ratings) serve as the reference.

In [ ]:
from tqdm import tqdm
tqdm.pandas()

def classify_text(text: str) -> str | None:
    """Classify a single review text; returns None on error."""
    try:
        result = classifier(str(text), truncation=True, max_length=512)[0]
        return stars_to_sentiment(result['label'])
    except Exception:
        return None

print("Classifying 300 reviews...")
df_sample['predicted_sentiment'] = df_sample['text'].progress_apply(classify_text)
print("Done.")

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

# Drop any rows where classification failed
df_eval = df_sample.dropna(subset=['predicted_sentiment'])

accuracy = accuracy_score(df_eval['true_sentiment'], df_eval['predicted_sentiment'])
print(f"Overall Accuracy: {accuracy:.3f}\n")
print(classification_report(df_eval['true_sentiment'], df_eval['predicted_sentiment']))

### Interpretation

| Class    | Observation |
|----------|-------------|
| Positive | Highest recall (94%) — model rarely misses positive reviews, but occasionally over-predicts (precision 67%) |
| Negative | Most balanced class — F1 0.77 |
| Neutral  | Hardest to classify — recall 46%. The boundary between 3 and 4 stars is inherently ambiguous |

71% accuracy without any fine-tuning on Amazon data represents a strong baseline. The neutral class performance suggests that fine-tuning on domain-specific data with a larger pool of verified neutral examples could yield meaningful improvement.

### Fine-tuning consideration
Fine-tuning the model on verified Amazon data is the logical next step. However, it requires a minimum of several hundred verified examples per class to be effective. With the current pool of 22 verified reviews, fine-tuning would risk overfitting. The human-in-the-loop pipeline is designed to collect this data over time, enabling fine-tuning once sufficient verified labels are available.

---
## 7. Quality Monitoring and Retraining Pipeline

The system monitors prediction quality by comparing model outputs against human-verified labels. When accuracy drops below the 70% threshold, the dashboard displays a retraining alert and allows the analyst to export verified data for fine-tuning.

In [ ]:
QUALITY_THRESHOLD = 0.70
MIN_VERIFIED = 10

def check_model_quality(db_path: str = DB_PATH) -> dict:
    """
    Assess model quality against human-verified labels stored in the database.

    Returns a status dict with keys: status, accuracy (if available),
    verified_count, and a human-readable message.
    """
    conn = sqlite3.connect(db_path)
    df = pd.read_sql(
        "SELECT sentiment, verified_sentiment FROM reviews WHERE verified_sentiment IS NOT NULL",
        conn
    )
    conn.close()

    if len(df) < MIN_VERIFIED:
        return {
            "status": "insufficient_data",
            "verified_count": len(df),
            "message": f"Need at least {MIN_VERIFIED} verified reviews, have {len(df)}"
        }

    accuracy = float((df['sentiment'] == df['verified_sentiment']).mean())
    needs_retraining = accuracy < QUALITY_THRESHOLD

    return {
        "status": "retrain_needed" if needs_retraining else "ok",
        "accuracy": round(accuracy, 3),
        "verified_count": len(df),
        "message": (
            f"Accuracy {accuracy:.1%} — retraining recommended"
            if needs_retraining
            else f"Accuracy {accuracy:.1%} — model performing well"
        )
    }

# Simulate verified labels using Amazon ground-truth
conn = sqlite3.connect(DB_PATH)
for _, row in df_eval.head(20).iterrows():
    conn.execute(
        "INSERT INTO reviews (text, sentiment, stars, confidence, verified_sentiment, timestamp) VALUES (?, ?, ?, ?, ?, ?)",
        (row['text'], row['predicted_sentiment'], 'N/A', 0.0,
         row['true_sentiment'], datetime.now().isoformat())
    )
conn.commit()
conn.close()

quality = check_model_quality()
print(quality)

In [ ]:
def export_training_data(db_path: str = DB_PATH, output_path: str = 'training_data_export.csv') -> dict:
    """
    Export human-verified reviews to CSV for model fine-tuning.

    Produces a two-column CSV (text, sentiment) containing all reviews
    that have a human-verified label. This file is intended for use in
    a cloud fine-tuning environment once enough verified data is available.
    """
    conn = sqlite3.connect(db_path)
    df = pd.read_sql(
        "SELECT text, verified_sentiment AS sentiment FROM reviews WHERE verified_sentiment IS NOT NULL",
        conn
    )
    conn.close()
    df.to_csv(output_path, index=False)
    return {"records": len(df), "path": output_path}

if quality.get('status') == 'retrain_needed':
    export = export_training_data()
    print(f"Training data exported: {export['records']} records → {export['path']}")
else:
    print("Model quality is acceptable — no export needed.")

---
## 8. Summary and Limitations

### What was built
An end-to-end sentiment analysis system consisting of:
- A pretrained BERT model classifying reviews into three sentiment classes
- A FastAPI REST API with six endpoints
- A SQLite database storing all predictions and human-verified labels
- A Streamlit dashboard for analysts
- A human-in-the-loop correction and quality monitoring workflow
- A training data export pipeline for future fine-tuning

### Evaluation results
Evaluated on a stratified sample of 300 Amazon reviews (100 per class):
- **Overall accuracy: 71.3%**
- Positive F1: 0.78 — Negative F1: 0.77 — Neutral F1: 0.55

### Key limitations
- **Neutral class performance**: Recall of 46% indicates the boundary between 3 and 4 stars is ambiguous. A model fine-tuned on domain-specific neutral examples would likely perform better.
- **No fine-tuning**: Fine-tuning requires a minimum of several hundred verified examples per class. The current human-in-the-loop pipeline is collecting this data, but the volume is not yet sufficient.
- **Retraining pipeline scope**: The current implementation monitors quality and exports verified data, but does not perform automated retraining. This is a deliberate design decision given the compute and data constraints of this project stage.
- **Single-server deployment**: The system is designed for local deployment. Production use would require containerisation (Docker), a persistent database server, and authentication on the API.